In [1]:
from pathlib import Path
from argparse import Namespace
import copy
import hashlib
import json
import math
import os
import pickle
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

ROOT = Path.cwd()

TRACE_ORIGIN = pd.Timestamp("2024-03-01 00:00:00")
PRIMARY_GPU = "A100-SXM4-80GB"
CAPACITY = 3456

Q_GRID = [0.50, 0.80, 0.90, 0.95, 0.99]
H_GRID = [1, 2, 4]
EVICTION_SLO = 0.10

RUN_CALIBRATION = True
RESUME = True




FORCE_WORKERS = None
WORKER_CAP = 6
RAM_PER_WORKER_GB = 3.0

def choose_workers():
    logical = os.cpu_count() or 2

    try:
        import psutil
        physical = psutil.cpu_count(logical=False) or max(1, logical // 2)
        available_gb = psutil.virtual_memory().available / (1024**3)

        cpu_limit = max(1, physical - 1)
        ram_limit = max(1, int(available_gb // RAM_PER_WORKER_GB))

        workers = min(WORKER_CAP, cpu_limit, ram_limit)

        info = {
            "logical_cpus": logical,
            "physical_cpus": physical,
            "available_ram_gb": round(available_gb, 2),
            "cpu_limit": cpu_limit,
            "ram_limit": ram_limit,
        }

    except Exception:
        workers = min(WORKER_CAP, max(1, logical // 2))
        info = {
            "logical_cpus": logical,
            "physical_cpus": None,
            "available_ram_gb": None,
            "cpu_limit": workers,
            "ram_limit": None,
        }

    if FORCE_WORKERS is not None:
        workers = max(1, int(FORCE_WORKERS))

    return workers, info

MAX_WORKERS, PARALLEL_INFO = choose_workers()

OUT = ROOT / "outputs_fame_gpu_04a_v3_parallel"
RUNS = ROOT / "fame_gpu_04a_v3_parallel_runs"
CHECKPOINT_DIR = RUNS / "checkpoints"
BRANCH_DIR = RUNS / "branches"

for p in [OUT, RUNS, CHECKPOINT_DIR, BRANCH_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("MAX_WORKERS:", MAX_WORKERS)
print("Parallel info:", PARALLEL_INFO)
print("Eviction SLO:", EVICTION_SLO)

Python: 3.13.14
MAX_WORKERS: 6
Parallel info: {'logical_cpus': 20, 'physical_cpus': 12, 'available_ram_gb': 49.53, 'cpu_limit': 11, 'ram_limit': 16}
Eviction SLO: 0.1


In [2]:
def locate_folder_or_zip(folder_name):
    for p in [ROOT/folder_name, Path("/mnt/data")/folder_name]:
        if p.exists() and p.is_dir():
            return p

    for zp in [ROOT/f"{folder_name}.zip", Path("/mnt/data")/f"{folder_name}.zip"]:
        if zp.exists():
            target = ROOT/f"{folder_name}_v2_extract"
            target.mkdir(exist_ok=True)
            with zipfile.ZipFile(zp) as z:
                z.extractall(target)
            p = target/folder_name
            if p.exists():
                return p

    raise FileNotFoundError(f"{folder_name}(.zip) não encontrado.")

OUT1 = locate_folder_or_zip("outputs_fame_gpu")
OUT3B = locate_folder_or_zip("outputs_fame_gpu_03b")

windows = pd.read_csv(
    OUT1/"fame_windows_7d.csv",
    parse_dates=[
        "D_start","D_end_exclusive",
        "C_start","C_end_exclusive",
        "E_start","E_end_exclusive"
    ]
)

binding = pd.read_csv(OUT3B/"binding_audit.csv")

quota_long = pd.read_csv(
    OUT3B/"candidate_quota_long.csv.gz",
    compression="gzip",
    parse_dates=["origin"]
)

display(windows)

,replication,D_start,D_end_exclusive,C_start,C_end_exclusive,E_start,E_end_exclusive
0,R01,2024-03-01,2024-04-26,2024-04-26,2024-05-03,2024-05-03,2024-05-10
1,R02,2024-03-01,2024-05-10,2024-05-10,2024-05-17,2024-05-17,2024-05-24
2,R03,2024-03-01,2024-05-24,2024-05-24,2024-05-31,2024-05-31,2024-06-07
3,R04,2024-03-01,2024-06-07,2024-06-07,2024-06-14,2024-06-14,2024-06-21
4,R05,2024-03-01,2024-06-21,2024-06-21,2024-06-28,2024-06-28,2024-07-05
5,R06,2024-03-01,2024-07-05,2024-07-05,2024-07-12,2024-07-12,2024-07-19
6,R07,2024-03-01,2024-07-19,2024-07-19,2024-07-26,2024-07-26,2024-08-02
7,R08,2024-03-01,2024-08-02,2024-08-02,2024-08-09,2024-08-09,2024-08-16
8,R09,2024-03-01,2024-08-16,2024-08-16,2024-08-23,2024-08-23,2024-08-30


In [3]:
c_binding = (
    binding[binding["period"]=="C"]
    .groupby("replication")["binding_share_offered"]
    .max()
    .rename("max_C_binding_share")
    .reset_index()
)

eligibility = windows[["replication"]].merge(
    c_binding,
    on="replication",
    how="left"
)

eligibility["decision_active_C"] = (
    eligibility["max_C_binding_share"].fillna(0) > 0
)

active_reps = eligibility.loc[
    eligibility["decision_active_C"],
    "replication"
].tolist()

inactive_reps = eligibility.loc[
    ~eligibility["decision_active_C"],
    "replication"
].tolist()

print("Decision-active:", active_reps)
print("Decision-inactive diagnostics:", inactive_reps)

eligibility.to_csv(
    OUT/"replication_eligibility.csv",
    index=False
)

display(eligibility)

Decision-active: ['R04', 'R05', 'R06', 'R07', 'R08', 'R09']
Decision-inactive diagnostics: ['R01', 'R02', 'R03']


,replication,max_C_binding_share,decision_active_C
0,R01,0.000000,False
1,R02,0.000000,False
2,R03,0.000000,False
3,R04,0.047619,True
4,R05,0.386905,True
5,R06,0.767857,True
6,R07,0.333333,True
7,R08,0.761905,True
8,R09,0.738095,True


In [4]:
DATA_DIR = None

for p in [ROOT/"data_fame_gpu", Path("/mnt/data")/"data_fame_gpu"]:
    if (p/"job_info_df.csv").exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError("Pasta data_fame_gpu não encontrada.")

jobs = pd.read_csv(DATA_DIR/"job_info_df.csv")
nodes = pd.read_csv(DATA_DIR/"node_info_df.csv")

a100_jobs = jobs[jobs["gpu_model"]==PRIMARY_GPU].copy()
a100_nodes = nodes[nodes["gpu_model"]==PRIMARY_GPU].copy()

a100_jobs["submit_time"] = pd.to_numeric(a100_jobs["submit_time"], errors="coerce")
a100_jobs["duration"] = pd.to_numeric(a100_jobs["duration"], errors="coerce")

hp_jobs = a100_jobs[a100_jobs["job_type"]=="HP"].copy()
spot_jobs = a100_jobs[a100_jobs["job_type"]=="Spot"].copy()

BASE_DATA = RUNS/"hp_only_data"
BASE_DATA.mkdir(exist_ok=True)

hp_jobs.to_csv(BASE_DATA/"job_info_df.csv", index=False)
a100_nodes.to_csv(BASE_DATA/"node_info_df.csv", index=False)

print("A100 HP jobs:", len(hp_jobs))
print("A100 Spot jobs:", len(spot_jobs))
print("A100 nodes:", len(a100_nodes))

A100 HP jobs: 137396
A100 Spot jobs: 31624
A100 nodes: 432


In [5]:
GFS_ORIGINAL = ROOT/"26ASPLOS-Spot"

if not GFS_ORIGINAL.exists():
    proc = subprocess.run(
        [
            "git","clone","--depth","1",
            "https://github.com/MachineLearningSystem/26ASPLOS-Spot.git",
            str(GFS_ORIGINAL)
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError("Falha ao obter 26ASPLOS-Spot.")

WORK_BASE = Path(tempfile.mkdtemp(prefix="fame_gpu_04a_v2_"))
GFS_FAME = WORK_BASE/"26ASPLOS-Spot-FAME"

shutil.copytree(
    GFS_ORIGINAL,
    GFS_FAME,
    ignore=shutil.ignore_patterns(".git","__pycache__","*.pyc",".pytest_cache")
)

print("Cópia de trabalho:", GFS_FAME)

Cópia de trabalho: C:\Users\tiago\AppData\Local\Temp\fame_gpu_04a_v2_2faespke\26ASPLOS-Spot-FAME


In [6]:
fame_policy_code = r"""
import pandas as pd

from .spot import SpotScheduler


class SilentLogger:
    def info(self, *args, **kwargs):
        return None
    def warning(self, *args, **kwargs):
        return None
    def error(self, *args, **kwargs):
        return None


class _NoOpEstimator:
    def train(self, *args, **kwargs):
        return None


class FAMEQuotaScheduler(SpotScheduler):

    def __init__(self, trace, cluster, args, log_dir, logger=None):
        if logger is None:
            logger = SilentLogger()

        super(FAMEQuotaScheduler, self).__init__(
            trace, cluster, args, log_dir, logger
        )

        self._name = "fame_quota"
        self.estimator = _NoOpEstimator()
        self.prev_index = 0
        self._quota_lookup = {}
        self._last_quota = {}

    def set_quota_lookup(self, lookup):
        self._quota_lookup = lookup
        self._last_quota = {}

    def update_spot_quota(self, org_request_df, now_timestamp):
        for vc_name in self.cluster.vc_dict:
            key = (int(self.time), str(vc_name))

            if key in self._quota_lookup:
                quota = self._quota_lookup[key]
                self._last_quota[vc_name] = quota
            elif vc_name in self._last_quota:
                quota = self._last_quota[vc_name]
            else:
                quota = 0.0

            capacity = self.cluster.vc_dict[vc_name].total_gpus
            quota = max(0.0, min(float(quota), float(capacity)))

            self.spot_quota[vc_name]["quota"][1] = quota
            self.spot_quota_record.append(
                [now_timestamp, vc_name, 1, quota, 1.0]
            )

    def simulate_until(self, stop_time, record=False):
        stop_time = int(stop_time)
        prev_index = int(getattr(self, "prev_index", 0))

        while self.time < stop_time:

            # 0. Quota / eviction-score updates
            if self.time % 300 == 0:
                if record and self.args.log_range[0] <= self.time <= self.args.log_range[1]:
                    now_timestamp = (
                        pd.Timestamp(self.args.trace_range[0])
                        + pd.Timedelta(seconds=self.time)
                    )
                    self.update_spot_quota(None, now_timestamp)

                if record:
                    for vc_name in self.guar_que_list:
                        vc = self.cluster.vc_dict[vc_name]
                        vc.update_eviction_score(self.time)

            # 1. Release completed jobs
            for vc_name in self.run_list:
                run_ls = self.run_list[vc_name].copy()
                vc = self.cluster.vc_dict[vc_name]

                for job in run_ls:
                    if (
                        job["type"] == "Spot"
                        and self.time - job.get_ckpt_time() >= job["spot_duration"]
                    ):
                        self.record_spot_succeed(job)
                        job.set_ckpt_time(self.time)

                    if job["remain"] == 0:
                        job["status"] = "end"
                        job["end_time"] = self.time
                        self.end_job_num += 1

                        assert vc.release_resource(job)
                        self.run_list[vc_name].remove(job)

                        self.gpu_request_dict[vc_name][job["type"]] -= job["gpu_request"]

                        if job["type"] == "Spot":
                            self.record_spot_succeed(job)
                        else:
                            self.guar_request_dict[job["org_key"]] -= job["gpu_request"]
                    else:
                        job["remain"] -= 1

            # 2A. Pend newly submitted jobs
            for idx in range(prev_index, self.total_job_num):
                job = self.trace.job_list[idx]

                if job["submit_time"] <= self.time:
                    self.pend_job(job)
                    prev_index = idx + 1
                else:
                    prev_index = idx
                    break

            self.prev_index = prev_index

            # 2B. HP queue
            for vc_name in self.guar_que_list:
                self.guar_que_list[vc_name].sort(
                    key=lambda x: (
                        -x.__getitem__("gpu_request"),
                        x.__getitem__("submit_time")
                    )
                )

                que_ls = self.guar_que_list[vc_name].copy()
                failed_gpu_num = 16

                for job in que_ls:
                    if job["gpu_request"] >= failed_gpu_num:
                        continue

                    if self.job_placer(job):
                        self.run_guar(job)

                    elif self.time >= self.args.log_range[0]:
                        place_flag, preempt_jobs = self.place_with_preemption(job)

                        if place_flag:
                            self.run_guar(job)
                            self.preempt(job, preempt_jobs)
                        else:
                            failed_gpu_num = job["gpu_request"]
                    else:
                        failed_gpu_num = job["gpu_request"]

            # 2C. Spot queue
            for key in self.spot_que_list:
                self.spot_que_list[key].sort(
                    key=lambda x: (
                        -x.__getitem__("gpu_request"),
                        x.__getitem__("worker_num"),
                        x.__getitem__("submit_time")
                    )
                )

                spot_que_ls = self.spot_que_list[key].copy()
                failed_gpu_num = 16

                for job in spot_que_ls:
                    if job["gpu_request"] >= failed_gpu_num:
                        continue

                    allowed = (
                        self.time > self.args.log_range[1]
                        or (
                            self.args.log_range[0] <= self.time <= self.args.log_range[1]
                            and (
                                self.gpu_request_dict[key]["Spot"]
                                + job["gpu_request"] * job["worker_num"]
                                <= self.spot_quota[key]["quota"][job["guarantee_request"]]
                            )
                        )
                    )

                    if allowed:
                        if self.job_placer(job):
                            self.run_spot(job)
                        else:
                            self.cluster.vc_dict[key].release_resource(job)
                            job.update({"nodes": []})
                            failed_gpu_num = job["gpu_request"]

            # 3. Recorder only in branches
            if record and self.time % 60 == 0:
                self.recorder.update_seq_recorder()

            if record and self.args.log_range[0] <= self.time <= self.args.log_range[1]:
                self.update_guar_gpu_time()

            self.time += 1

        self.prev_index = prev_index
        return True
"""

(GFS_FAME/"policy"/"fame_quota.py").write_text(
    fame_policy_code,
    encoding="utf-8"
)

7021

In [7]:
def replace_once(path, old, new, label):
    txt = path.read_text(encoding="utf-8")
    if old not in txt:
        raise RuntimeError(f"Patch não encontrado: {label}")
    path.write_text(txt.replace(old,new,1), encoding="utf-8")


spot_path = GFS_FAME/"policy"/"spot.py"
spot_txt = spot_path.read_text(encoding="utf-8")
spot_txt = spot_txt.replace(
    "from estimator.gpu_request_estimator import GPURequestEstimator\n",
    ""
)
spot_txt = spot_txt.replace(
    "self.estimator = GPURequestEstimator(args)",
    "self.estimator = None"
)
spot_path.write_text(spot_txt, encoding="utf-8")


(GFS_FAME/"policy"/"__init__.py").write_text(
    "from .fame_quota import FAMEQuotaScheduler, SilentLogger\n",
    encoding="utf-8"
)


policy_path = GFS_FAME/"policy"/"policy.py"
replace_once(
    policy_path,
    'if self.args.scheduler == "spot_scheduler":',
    'if self.args.scheduler in ["spot_scheduler", "fame_quota"]:',
    "Spot placer"
)


utils_path = GFS_FAME/"utils.py"
utils_txt = utils_path.read_text(encoding="utf-8")

imp0 = utils_txt.find("from policy import (")
if imp0 != -1:
    imp1 = utils_txt.find(")\n", imp0)
    if imp1 == -1:
        raise RuntimeError("Fim do import policy não localizado.")
    utils_txt = (
        utils_txt[:imp0]
        + "from policy import FAMEQuotaScheduler\n"
        + utils_txt[imp1+2:]
    )


sim0 = utils_txt.find("def simulate_vc(")
trace0 = utils_txt.find("# 任务trace数据", sim0)

if sim0 != -1 and trace0 != -1:
    utils_txt = utils_txt[:sim0] + utils_txt[trace0:]

utils_path.write_text(utils_txt, encoding="utf-8")


for p in [
    GFS_FAME/"policy"/"fame_quota.py",
    GFS_FAME/"policy"/"spot.py",
    GFS_FAME/"policy"/"policy.py",
    GFS_FAME/"utils.py",
]:
    proc = subprocess.run(
        [sys.executable,"-m","py_compile",str(p)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    if proc.returncode != 0:
        raise RuntimeError(f"{p.name}: {proc.stdout}")

print("Patches OK.")

Patches OK.


In [8]:

for mod in list(sys.modules):
    if (
        mod == "utils"
        or mod == "cluster"
        or mod == "job"
        or mod == "policy"
        or mod.startswith("policy.")
    ):
        del sys.modules[mod]

sys.path.insert(0, str(GFS_FAME))

import utils
import cluster as gfs_cluster

from job import Job, Trace
from policy.fame_quota import FAMEQuotaScheduler, SilentLogger
from policy.recorder.recorder import Recorder

print("Imports OK.")

Imports OK.


In [9]:
last_active = windows[
    windows["replication"].isin(active_reps)
].copy()

last_C_end = last_active["C_end_exclusive"].max()

date_range = (
    str(TRACE_ORIGIN),
    str(last_C_end - pd.Timedelta(seconds=1))
)


base_df = utils.trace_process(
    str(BASE_DATA),
    date_range,
    (10**12, 10**12 + 1)
)

base_trace = utils.trace_parser(base_df)
base_cluster = gfs_cluster.Cluster(
    str(BASE_DATA/"node_info_df.csv")
)

last_end_sec = int(
    (last_C_end - TRACE_ORIGIN).total_seconds()
)

base_args = Namespace(
    scheduler="fame_quota",
    log_range=(last_end_sec+1, last_end_sec+2),
    trace_range=(str(TRACE_ORIGIN), str(last_C_end)),
    guarantee_hour=[1],
    guarantee_rate=0.90,
    pred_len=4,
    ckpt_interval=300,
    use_gpu=False,
)

base_scheduler = FAMEQuotaScheduler(
    base_trace,
    base_cluster,
    base_args,
    str(RUNS/"base"),
    SilentLogger()
)

print("Base jobs:", base_scheduler.total_job_num)
print("Base time:", base_scheduler.time)

C:\Users\tiago\AppData\Local\Temp\fame_gpu_04a_v2_2faespke\26ASPLOS-Spot-FAME\utils.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


Base jobs: 131817
Base time: 0


In [10]:
branch_worker_source = r"""
from pathlib import Path
import argparse
import json
import pickle
import sys

import numpy as np
import pandas as pd


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--spot-csv", required=True)
    parser.add_argument("--quota-csv", required=True)
    parser.add_argument("--start-sec", type=int, required=True)
    parser.add_argument("--end-sec", type=int, required=True)
    parser.add_argument("--out-json", required=True)
    parser.add_argument("--gfs-dir", required=True)
    args_cli = parser.parse_args()

    sys.path.insert(0, args_cli.gfs_dir)

    from job import Job
    from policy.fame_quota import SilentLogger
    from policy.recorder.recorder import Recorder

    with open(args_cli.checkpoint, "rb") as f:
        sched = pickle.load(f)

    start_sec = int(args_cli.start_sec)
    end_sec = int(args_cli.end_sec)

    # Keep HP jobs only through end of calibration.
    sched.trace.job_list = [
        j for j in sched.trace.job_list
        if float(j["submit_time"]) < end_sec
    ]

    # Add fresh Spot jobs from calibration period.
    spot_df = pd.read_csv(args_cli.spot_csv)

    new_spot_jobs = []

    for row_idx, row in spot_df.iterrows():
        s = row.copy()

        s["job_index"] = int(10_000_000 + row_idx)
        s["vc_name"] = s["gpu_model"]
        s["type"] = "Spot"
        s["org_key"] = (
            str(s["gpu_model"])
            + "&&cluster&&"
            + str(s["organization"])
        )

        s["remain"] = s["duration"]
        s["start_time"] = sys.maxsize
        s["end_time"] = sys.maxsize
        s["ckpt_times"] = 0
        s["preempt_times"] = 0
        s["queue"] = 0
        s["jct"] = 0
        s["guarantee_request"] = 1
        s["status"] = None

        new_spot_jobs.append(Job(s))

    sched.trace.job_list.extend(new_spot_jobs)
    sched.trace.sort_jobs("submit_time")

    sched.total_job_num = len(sched.trace.job_list)

    # The checkpoint stops BEFORE processing start_sec.
    sched.prev_index = sum(
        float(j["submit_time"]) < start_sec
        for j in sched.trace.job_list
    )

    sched.end_job_num = sum(
        j.get("status") == "end"
        for j in sched.trace.job_list
    )

    # Period-specific settings.
    sched.args.log_range = (start_sec, end_sec - 1)
    sched.timespan = end_sec - start_sec
    sched.args.scheduler = "fame_quota"
    sched.args.guarantee_hour = [1]

    # Reset period metrics. Warm-up state remains intact.
    sched.spot_failed = sched.init_count_dict()
    sched.spot_succeed = sched.init_count_dict()
    sched.preemption_times = 0
    sched.spot_quota_record = []
    sched.guar_gpu_time = sched.init_count_dict()
    sched.spot_gpu_time = sched.init_count_dict()

    for vc in sched.spot_que_list:
        sched.spot_que_list[vc] = []
        sched.gpu_request_dict[vc]["Spot"] = 0

    sched.spot_quota = sched.init_spot_quota_dict()

    sched.logger = SilentLogger()
    sched.recorder = Recorder(
        sched,
        str(Path(args_cli.out_json).parent)
    )
    sched.recorder.logger = sched.logger

    qdf = pd.read_csv(args_cli.quota_csv)
    lookup = {
        (int(r.time), str(r.vc_name)): float(r.quota)
        for r in qdf.itertuples(index=False)
    }

    sched.set_quota_lookup(lookup)

    # Exactly one calibration week.
    sched.simulate_until(end_sec, record=True)

    if len(sched.recorder.record_df_list) == 0:
        raise RuntimeError("Recorder vazio.")

    record_df = pd.concat(
        sched.recorder.record_df_list,
        ignore_index=True
    )

    allocation = (
        (
            record_df["total_gpu_num"]
            - record_df["idle_gpu_num"]
        )
        / record_df["total_gpu_num"]
    )

    vc_name = next(iter(sched.cluster.vc_dict.keys()))

    failed = float(sched.spot_failed[vc_name])
    succeed = float(sched.spot_succeed[vc_name])

    native_eviction_rate = (
        failed / (failed + succeed)
        if failed + succeed > 0
        else 0.0
    )

    # Job-level secondary diagnostics.
    spot_period = [
        j for j in sched.trace.job_list
        if (
            j["type"] == "Spot"
            and start_sec <= float(j["submit_time"]) < end_sec
        )
    ]

    hp_period = [
        j for j in sched.trace.job_list
        if (
            j["type"] == "HP"
            and start_sec <= float(j["submit_time"]) < end_sec
        )
    ]

    spot_started = [
        j for j in spot_period
        if j.get("status") in {"run","end"}
    ]

    hp_started = [
        j for j in hp_period
        if j.get("status") in {"run","end"}
    ]

    qvals = qdf["quota"].to_numpy(dtype=float)

    result = {
        "status": "ok",
        "n_HP_submitted": len(hp_period),
        "n_Spot_submitted": len(spot_period),
        "n_HP_started_by_end": len(hp_started),
        "n_Spot_started_by_end": len(spot_started),

        "allocation_rate": float(allocation.mean()),
        "allocation_rate_min": float(allocation.min()),
        "allocation_rate_max": float(allocation.max()),

        "native_spot_failed": failed,
        "native_spot_succeed": succeed,
        "eviction_rate": float(native_eviction_rate),

        "spot_jobs_preempted_share": (
            float(
                np.mean([
                    j["preempt_times"] > 0
                    for j in spot_period
                ])
            )
            if spot_period else 0.0
        ),

        "mean_HP_queue_jobs": float(record_df["guar_que_job_num"].mean()),
        "max_HP_queue_jobs": float(record_df["guar_que_job_num"].max()),
        "mean_Spot_queue_jobs": float(record_df["spot_que_job_num"].mean()),
        "max_Spot_queue_jobs": float(record_df["spot_que_job_num"].max()),

        "mean_HP_queue_s_started": (
            float(np.mean([j["queue"] for j in hp_started]))
            if hp_started else np.nan
        ),

        "mean_Spot_queue_s_started": (
            float(np.mean([j["queue"] for j in spot_started]))
            if spot_started else np.nan
        ),

        "guar_useful_gpu_time_rate": float(
            sum(sched.guar_gpu_time.values())
            / (sched.timespan * sched.cluster.total_gpus)
        ),

        "spot_useful_gpu_time_rate": float(
            sum(sched.spot_gpu_time.values())
            / (sched.timespan * sched.cluster.total_gpus)
        ),

        "quota_min": float(qvals.min()),
        "quota_mean": float(qvals.mean()),
        "quota_max": float(qvals.max()),
    }

    with open(args_cli.out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)


if __name__ == "__main__":
    main()
"""

worker_path = GFS_FAME/"branch_worker.py"
worker_path.write_text(
    branch_worker_source,
    encoding="utf-8"
)

proc = subprocess.run(
    [sys.executable,"-m","py_compile",str(worker_path)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

if proc.returncode != 0:
    raise RuntimeError(proc.stdout)

print("Worker:", worker_path)

Worker: C:\Users\tiago\AppData\Local\Temp\fame_gpu_04a_v2_2faespke\26ASPLOS-Spot-FAME\branch_worker.py


In [11]:
grid = pd.MultiIndex.from_product(
    [Q_GRID,H_GRID],
    names=["q","H"]
).to_frame(index=False)

def period_row(rep):
    return windows[windows["replication"]==rep].iloc[0]

def sec(ts):
    return int((pd.Timestamp(ts)-TRACE_ORIGIN).total_seconds())

def candidate_hourly(rep, period, q, H):
    z = quota_long[
        (quota_long["replication"]==rep)
        & (quota_long["period"]==period)
        & np.isclose(quota_long["q"],q)
        & (quota_long["H"]==H)
    ].copy()

    return z[["origin","quota"]].sort_values("origin")

def expand_5min(hourly_df):
    rows=[]

    for r in hourly_df.itertuples(index=False):
        for minute in range(0,60,5):
            ts = pd.Timestamp(r.origin) + pd.Timedelta(minutes=minute)

            rows.append({
                "time": sec(ts),
                "vc_name": PRIMARY_GPU,
                "quota": float(np.clip(r.quota,0,CAPACITY))
            })

    return pd.DataFrame(rows)

def spot_csv_for_rep(rep):
    wr = period_row(rep)
    s0,s1 = sec(wr["C_start"]),sec(wr["C_end_exclusive"])

    z = spot_jobs[
        (spot_jobs["submit_time"]>=s0)
        & (spot_jobs["submit_time"]<s1)
    ].copy()

    p = RUNS/f"{rep}_spot_C.csv"
    z.to_csv(p,index=False)
    return p

def save_checkpoint(scheduler, path):
    
    scheduler.logger = SilentLogger()
    scheduler.recorder = Recorder(
        scheduler,
        str(CHECKPOINT_DIR)
    )
    scheduler.recorder.logger = scheduler.logger

    with open(path,"wb") as f:
        pickle.dump(
            scheduler,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    
    with open(path,"rb") as f:
        test = pickle.load(f)

    assert test.time == scheduler.time
    return path.stat().st_size

print("Grid size:",len(grid))

Grid size: 15


In [12]:
RESULT_CSV = OUT/"calibration_candidate_results.csv"
BASE_STATE_JSON = RUNS/"base_state.json"
BASE_LATEST_PKL = CHECKPOINT_DIR/"base_latest.pkl"

if RESULT_CSV.exists() and RESUME:
    results = pd.read_csv(RESULT_CSV)
else:
    results = pd.DataFrame()

completed_reps = []

if not results.empty:
    for rep in active_reps:
        g = results[
            (results["replication"]==rep)
            & (results["status"]=="ok")
        ]
        if len(g)==len(grid):
            completed_reps.append(rep)

print("Replicações já completas:",completed_reps)

Replicações já completas: []


In [13]:
def build_fresh_base():
    bt = utils.trace_parser(base_df.copy())

    bc = gfs_cluster.Cluster(
        str(BASE_DATA/"node_info_df.csv")
    )

    bs = FAMEQuotaScheduler(
        bt,
        bc,
        copy.deepcopy(base_args),
        str(RUNS/"base"),
        SilentLogger()
    )

    return bs


def checkpoint_path_for(rep):
    return CHECKPOINT_DIR/f"{rep}_HP_checkpoint.pkl"


def checkpoint_is_valid(path, expected_time):
    if not path.exists():
        return False

    try:
        with open(path, "rb") as f:
            sched = pickle.load(f)
        return int(sched.time) == int(expected_time)
    except Exception:
        return False



valid_existing = []

for rep in active_reps:
    wr = period_row(rep)
    expected = sec(wr["C_start"])
    p = checkpoint_path_for(rep)

    if checkpoint_is_valid(p, expected):
        valid_existing.append((rep, expected, p))

if valid_existing:
    latest_rep, latest_time, latest_path = valid_existing[-1]

    with open(latest_path, "rb") as f:
        base_scheduler = pickle.load(f)

    print(
        "Retomando warm-up do checkpoint",
        latest_rep,
        "time=",
        base_scheduler.time
    )
else:
    latest_rep = None
    base_scheduler = build_fresh_base()
    print("Iniciando warm-up em 01/03/2024.")

Iniciando warm-up em 01/03/2024.


In [14]:
checkpoint_runtime = []

for rep in active_reps:
    wr = period_row(rep)
    target_time = sec(wr["C_start"])
    cp = checkpoint_path_for(rep)

    if checkpoint_is_valid(cp, target_time):
        print(
            "CHECKPOINT OK/SKIP",
            rep,
            "time=",
            target_time
        )
        continue

    if base_scheduler.time > target_time:
        
        
        print(
            f"Checkpoint {rep} ausente, mas base atual é posterior. "
            "Reconstruindo warm-up do início."
        )
        base_scheduler = build_fresh_base()

    t0 = time.time()

    base_scheduler.simulate_until(
        target_time,
        record=False
    )

    elapsed = time.time() - t0

    cp_bytes = save_checkpoint(
        base_scheduler,
        cp
    )

    checkpoint_runtime.append({
        "replication": rep,
        "checkpoint_time": target_time,
        "warmup_increment_s": elapsed,
        "checkpoint_MB": cp_bytes / 1024**2,
    })

    pd.DataFrame(checkpoint_runtime).to_csv(
        OUT/"checkpoint_runtime.csv",
        index=False
    )

    print(
        rep,
        "checkpoint criado |",
        round(cp_bytes/1024**2, 1),
        "MB | warm-up incremental:",
        round(elapsed/60, 2),
        "min"
    )

print("\\nTodos os checkpoints necessários estão disponíveis.")

R04 checkpoint criado | 25.6 MB | warm-up incremental: 26.37 min
R05 checkpoint criado | 25.8 MB | warm-up incremental: 4.36 min
R06 checkpoint criado | 26.0 MB | warm-up incremental: 2.1 min
R07 checkpoint criado | 26.9 MB | warm-up incremental: 2.06 min
R08 checkpoint criado | 27.2 MB | warm-up incremental: 1.92 min
R09 checkpoint criado | 27.4 MB | warm-up incremental: 4.39 min
\nTodos os checkpoints necessários estão disponíveis.


In [15]:




RESULT_CSV = OUT/"calibration_candidate_results.csv"

if RESULT_CSV.exists() and RESUME:
    results = pd.read_csv(RESULT_CSV)
else:
    results = pd.DataFrame()


def candidate_done(rep, q, H):
    if results.empty:
        return False

    z = results[
        (results["replication"] == rep)
        & np.isclose(results["q"], q)
        & (results["H"] == H)
        & (results["status"] == "ok")
    ]

    return len(z) > 0


def decision_hash(schedule_5min):
    
    
    decision_path = np.floor(
        schedule_5min["quota"].to_numpy(dtype=float)
    ).astype(np.int32)

    return hashlib.sha256(
        decision_path.tobytes()
    ).hexdigest()


def prepare_equivalence_classes():
    eq_rows = []
    class_payloads = []

    for rep in active_reps:
        cp = checkpoint_path_for(rep)

        if not cp.exists():
            raise FileNotFoundError(
                f"Checkpoint ausente: {cp}"
            )

        wr = period_row(rep)
        s0 = sec(wr["C_start"])
        s1 = sec(wr["C_end_exclusive"])
        spot_path = spot_csv_for_rep(rep)

        local = []

        for rr in grid.itertuples(index=False):
            q, H = float(rr.q), int(rr.H)

            sched = expand_5min(
                candidate_hourly(rep, "C", q, H)
            )

            hsh = decision_hash(sched)

            local.append({
                "replication": rep,
                "q": q,
                "H": H,
                "decision_path_hash": hsh,
                "quota_min": float(sched["quota"].min()),
                "quota_mean": float(sched["quota"].mean()),
                "quota_max": float(sched["quota"].max()),
                "schedule": sched,
                "checkpoint": cp,
                "spot_csv": spot_path,
                "start_sec": s0,
                "end_sec": s1,
            })

        
        local_df = pd.DataFrame([
            {k:v for k,v in d.items() if k != "schedule"}
            for d in local
        ])

        for hsh, g in local_df.groupby("decision_path_hash"):
            members = [
                d for d in local
                if d["decision_path_hash"] == hsh
            ]

            
            members = sorted(
                members,
                key=lambda d: (d["q"], d["H"])
            )

            representative = members[0]

            class_id = f"{rep}_{hsh[:12]}"

            for d in members:
                eq_rows.append({
                    "replication": rep,
                    "q": d["q"],
                    "H": d["H"],
                    "decision_path_hash": hsh,
                    "class_id": class_id,
                    "representative_q": representative["q"],
                    "representative_H": representative["H"],
                    "class_size": len(members),
                    "quota_min": d["quota_min"],
                    "quota_mean": d["quota_mean"],
                    "quota_max": d["quota_max"],
                })

            class_payloads.append({
                "class_id": class_id,
                "replication": rep,
                "hash": hsh,
                "members": members,
                "representative": representative,
            })

    eq_map = pd.DataFrame(eq_rows)

    eq_map.to_csv(
        OUT/"candidate_equivalence_map.csv",
        index=False
    )

    return eq_map, class_payloads


eq_map, equivalence_classes = prepare_equivalence_classes()

print(
    "Candidatos totais:",
    len(active_reps) * len(grid)
)

print(
    "Classes de decisão únicas:",
    len(equivalence_classes)
)

print(
    "Simulações evitadas por equivalência exata:",
    len(active_reps) * len(grid) - len(equivalence_classes)
)

display(
    eq_map.groupby("replication").agg(
        candidates=("q", "size"),
        unique_decision_paths=("class_id", "nunique"),
        max_equivalence_class=("class_size", "max"),
    )
)


def persist_result(row):
    global results

    if not results.empty:
        mask = (
            (results["replication"] == row["replication"])
            & np.isclose(results["q"], row["q"])
            & (results["H"] == row["H"])
        )
        results = results.loc[~mask].copy()

    results = pd.concat(
        [results, pd.DataFrame([row])],
        ignore_index=True
    )

    results.to_csv(
        RESULT_CSV,
        index=False
    )


def class_already_available(cls):
    
    if results.empty:
        return None

    for m in cls["members"]:
        z = results[
            (results["replication"] == m["replication"])
            & np.isclose(results["q"], m["q"])
            & (results["H"] == m["H"])
            & (results["status"] == "ok")
        ]

        if len(z):
            return z.iloc[-1].to_dict()

    return None


def build_task(cls):
    rep = cls["replication"]
    m = cls["representative"]

    rdir = BRANCH_DIR/cls["class_id"]
    rdir.mkdir(exist_ok=True)

    quota_path = rdir/"quota.csv"
    m["schedule"].to_csv(
        quota_path,
        index=False
    )

    out_json = rdir/"result.json"

    return {
        "class_id": cls["class_id"],
        "replication": rep,
        "q": m["q"],
        "H": m["H"],
        "run_id": (
            f"{rep}_C_q{m['q']:.2f}_H{m['H']}"
        ),
        "checkpoint": m["checkpoint"],
        "spot_csv": m["spot_csv"],
        "quota_csv": quota_path,
        "start_sec": m["start_sec"],
        "end_sec": m["end_sec"],
        "out_json": out_json,
    }


def execute_one_branch(job):
    t0 = time.time()

    cmd = [
        sys.executable,
        str(GFS_FAME/"branch_worker.py"),
        "--checkpoint", str(job["checkpoint"]),
        "--spot-csv", str(job["spot_csv"]),
        "--quota-csv", str(job["quota_csv"]),
        "--start-sec", str(job["start_sec"]),
        "--end-sec", str(job["end_sec"]),
        "--out-json", str(job["out_json"]),
        "--gfs-dir", str(GFS_FAME),
    ]

    child_env = os.environ.copy()

    
    for key in [
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ]:
        child_env[key] = "1"

    child_env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.run(
        cmd,
        cwd=GFS_FAME,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=child_env,
    )

    elapsed = time.time() - t0

    row = {
        "replication": job["replication"],
        "period": "C",
        "q": job["q"],
        "H": job["H"],
        "run_id": job["run_id"],
        "return_code": proc.returncode,
        "elapsed_s": elapsed,
        "decision_path_hash": None,
        "execution_mode": "simulated",
    }

    if proc.returncode == 0 and job["out_json"].exists():
        with open(
            job["out_json"],
            "r",
            encoding="utf-8"
        ) as f:
            met = json.load(f)

        row.update(met)
        row["status"] = "ok"

    else:
        row["status"] = "error"
        row["console_tail"] = proc.stdout[-4000:]

    return row


def propagate_class_result(cls, source_row):
    
    
    for m in cls["members"]:
        row = dict(source_row)

        row["replication"] = m["replication"]
        row["period"] = "C"
        row["q"] = m["q"]
        row["H"] = m["H"]
        row["decision_path_hash"] = cls["hash"]

        row["quota_min"] = m["quota_min"]
        row["quota_mean"] = m["quota_mean"]
        row["quota_max"] = m["quota_max"]

        is_rep = (
            np.isclose(m["q"], cls["representative"]["q"])
            and m["H"] == cls["representative"]["H"]
        )

        row["execution_mode"] = (
            "simulated"
            if is_rep
            else "decision_equivalent_reuse"
        )

        row["source_q"] = cls["representative"]["q"]
        row["source_H"] = cls["representative"]["H"]
        row["equivalence_class_size"] = len(cls["members"])

        if not is_rep:
            row["elapsed_s"] = 0.0
            row["run_id"] = (
                f"{m['replication']}_C_"
                f"q{m['q']:.2f}_H{m['H']}_"
                "reused"
            )

        persist_result(row)



pending_classes = []

for cls in equivalence_classes:
    available = class_already_available(cls)

    if available is not None:
        propagate_class_result(
            cls,
            available
        )
    else:
        pending_classes.append(cls)

print(
    "\\nClasses únicas pendentes:",
    len(pending_classes)
)

tasks = [
    (cls, build_task(cls))
    for cls in pending_classes
]




parallel_runtime_rows = []
session_start = time.time()
durations = []

if RUN_CALIBRATION and tasks:

    print(
        f"\\nIniciando {len(tasks)} branches únicos "
        f"com MAX_WORKERS={MAX_WORKERS}."
    )

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        future_map = {
            executor.submit(
                execute_one_branch,
                task
            ): cls
            for cls, task in tasks
        }

        n_total = len(future_map)
        n_done = 0

        for future in as_completed(future_map):
            cls = future_map[future]
            n_done += 1

            try:
                row = future.result()
            except Exception as exc:
                row = {
                    "replication": cls["replication"],
                    "period": "C",
                    "q": cls["representative"]["q"],
                    "H": cls["representative"]["H"],
                    "status": "error",
                    "return_code": -999,
                    "elapsed_s": np.nan,
                    "console_tail": repr(exc),
                    "execution_mode": "simulated",
                }

            if row["status"] == "ok":
                row["decision_path_hash"] = cls["hash"]

                propagate_class_result(
                    cls,
                    row
                )

                durations.append(
                    float(row["elapsed_s"])
                )

            else:
                
                row["decision_path_hash"] = cls["hash"]
                persist_result(row)

            elapsed_wall = time.time() - session_start

            mean_branch = (
                np.mean(durations)
                if durations
                else np.nan
            )

            
            remaining = n_total - n_done

            eta_s = (
                remaining
                * mean_branch
                / MAX_WORKERS
                if np.isfinite(mean_branch)
                else np.nan
            )

            parallel_runtime_rows.append({
                "completed_unique_classes": n_done,
                "total_unique_classes": n_total,
                "wall_elapsed_s": elapsed_wall,
                "mean_successful_branch_s": mean_branch,
                "approx_eta_s": eta_s,
                "workers": MAX_WORKERS,
            })

            pd.DataFrame(
                parallel_runtime_rows
            ).to_csv(
                OUT/"parallel_runtime_progress.csv",
                index=False
            )

            print(
                f"[{n_done}/{n_total}] "
                f"{cls['class_id']} -> {row['status']} | "
                f"branch={row.get('elapsed_s', np.nan)/60:.2f} min | "
                f"wall={elapsed_wall/3600:.2f} h | "
                f"ETA≈{eta_s/3600:.2f} h"
                if np.isfinite(eta_s)
                else
                f"[{n_done}/{n_total}] "
                f"{cls['class_id']} -> {row['status']}"
            )

elif not tasks:
    print("Todos os branches já estão disponíveis.")

else:
    print("RUN_CALIBRATION=False")

Candidatos totais: 90
Classes de decisão únicas: 90
Simulações evitadas por equivalência exata: 0


,candidates,unique_decision_paths,max_equivalence_class
replication,,,
R04,15,15,1
R05,15,15,1
R06,15,15,1
R07,15,15,1
R08,15,15,1
R09,15,15,1


\nClasses únicas pendentes: 90
\nIniciando 90 branches únicos com MAX_WORKERS=6.
[1/90] R04_193123e3d687 -> ok | branch=8.50 min | wall=0.14 h | ETA≈2.10 h
[2/90] R04_43aa0020545c -> ok | branch=8.50 min | wall=0.14 h | ETA≈2.08 h
[3/90] R04_42f5afa7954a -> ok | branch=8.51 min | wall=0.14 h | ETA≈2.05 h
[4/90] R04_0a0ff0b3c8de -> ok | branch=8.51 min | wall=0.14 h | ETA≈2.03 h
[5/90] R04_027477704dbf -> ok | branch=8.51 min | wall=0.14 h | ETA≈2.01 h
[6/90] R04_0690f558edd6 -> ok | branch=8.53 min | wall=0.14 h | ETA≈1.99 h
[7/90] R04_5b0837400d9c -> ok | branch=7.24 min | wall=0.26 h | ETA≈1.92 h
[8/90] R04_610159a1c21d -> ok | branch=7.27 min | wall=0.26 h | ETA≈1.87 h
[9/90] R04_a3e6acaae280 -> ok | branch=7.28 min | wall=0.26 h | ETA≈1.82 h
[10/90] R04_68f4beeddd3c -> ok | branch=7.29 min | wall=0.26 h | ETA≈1.78 h
[11/90] R04_839823b9a795 -> ok | branch=7.48 min | wall=0.27 h | ETA≈1.75 h
[12/90] R04_a6a72172c858 -> ok | branch=7.48 min | wall=0.27 h | ETA≈1.72 h
[13/90] R04_d00b

In [16]:
if not RESULT_CSV.exists():
    raise FileNotFoundError(
        "calibration_candidate_results.csv ainda não existe."
    )

results = pd.read_csv(RESULT_CSV)

completion = (
    results[results["status"]=="ok"]
    .groupby("replication")
    .size()
    .reindex(active_reps, fill_value=0)
    .rename("n_ok")
    .reset_index()
)

completion["expected"] = len(grid)
completion["complete"] = (
    completion["n_ok"] == len(grid)
)

display(completion)

simulated_count = int(
    (
        results.get(
            "execution_mode",
            pd.Series(index=results.index, dtype=object)
        )
        == "simulated"
    ).sum()
)

reuse_count = int(
    (
        results.get(
            "execution_mode",
            pd.Series(index=results.index, dtype=object)
        )
        == "decision_equivalent_reuse"
    ).sum()
)

print("Candidatos simulados diretamente:", simulated_count)
print("Candidatos reutilizados por equivalência exata:", reuse_count)

if not completion["complete"].all():
    print(
        "Há candidatos pendentes ou com erro. "
        "Reexecute com RESUME=True."
    )

,replication,n_ok,expected,complete
0,R04,15,15,True
1,R05,15,15,True
2,R06,15,15,True
3,R07,15,15,True
4,R08,15,15,True
5,R09,15,15,True


Candidatos simulados diretamente: 90
Candidatos reutilizados por equivalência exata: 0


In [17]:
ok=results[results["status"]=="ok"].copy()

ok["slo_ok"]=ok["eviction_rate"]<=EVICTION_SLO

ok["utility"]=np.where(
    ok["slo_ok"],
    ok["allocation_rate"],
    0.0
)

selected_rows=[]

for rep,g in ok.groupby("replication"):

    feasible=g[g["slo_ok"]].copy()

    if len(feasible):
        feasible["ref_distance"]=(
            abs(feasible["q"]-.90)
            + .05*abs(feasible["H"]-1)
        )

        chosen=(
            feasible
            .sort_values(
                [
                    "allocation_rate",
                    "eviction_rate",
                    "ref_distance"
                ],
                ascending=[False,True,True]
            )
            .iloc[0]
        )

        reason="max_allocation_subject_to_native_eviction_10pct"

    else:
        chosen=(
            g.sort_values(
                ["eviction_rate","allocation_rate"],
                ascending=[True,False]
            )
            .iloc[0]
        )

        reason="fallback_min_native_eviction"

    d=chosen.to_dict()
    d["selection_reason"]=reason
    selected_rows.append(d)

selected=pd.DataFrame(selected_rows)

cols=[
    "replication","q","H",
    "allocation_rate","eviction_rate",
    "mean_HP_queue_jobs","max_HP_queue_jobs",
    "mean_Spot_queue_jobs",
    "spot_useful_gpu_time_rate",
    "utility","selection_reason"
]

display(selected[cols])

selected.to_csv(
    OUT/"doc_selected_C.csv",
    index=False
)

,replication,q,H,allocation_rate,eviction_rate,mean_HP_queue_jobs,max_HP_queue_jobs,mean_Spot_queue_jobs,spot_useful_gpu_time_rate,utility,selection_reason
0,R04,0.90,1,0.752675,0.000000,0.000000,0.0,0.000000,0.003468,0.752675,max_allocation_subject_to_native_eviction_10pct
1,R05,0.95,1,0.903924,0.002488,0.000000,0.0,0.573611,0.003884,0.903924,max_allocation_subject_to_native_eviction_10pct
2,R06,0.80,4,0.923889,0.002806,0.000000,0.0,0.949306,0.004851,0.923889,max_allocation_subject_to_native_eviction_10pct
3,R07,0.90,4,0.785112,0.000308,0.000000,0.0,0.022619,0.002041,0.785112,max_allocation_subject_to_native_eviction_10pct
4,R08,0.90,2,0.856691,0.005488,0.387302,16.0,21.759226,0.003008,0.856691,max_allocation_subject_to_native_eviction_10pct
5,R09,0.90,2,0.919637,0.008099,5.697024,68.0,41.659524,0.002340,0.919637,max_allocation_subject_to_native_eviction_10pct


In [18]:
diag=[]

for rep,g in ok.groupby("replication"):

    diag.append({
        "replication":rep,
        "n_candidates":len(g),
        "n_SLO_ok":int(g["slo_ok"].sum()),
        "utility_range":float(g["utility"].max()-g["utility"].min()),
        "allocation_range":float(g["allocation_rate"].max()-g["allocation_rate"].min()),
        "eviction_min":float(g["eviction_rate"].min()),
        "eviction_max":float(g["eviction_rate"].max()),
        "mean_HP_queue_min":float(g["mean_HP_queue_jobs"].min()),
        "mean_HP_queue_max":float(g["mean_HP_queue_jobs"].max()),
        "surface_identified":bool(
            (g["utility"].max()-g["utility"].min())>1e-8
            or
            (g["allocation_rate"].max()-g["allocation_rate"].min())>1e-8
            or
            (g["eviction_rate"].max()-g["eviction_rate"].min())>1e-8
        ),
    })

diag=pd.DataFrame(diag)

display(diag)

diag.to_csv(
    OUT/"calibration_diagnostics.csv",
    index=False
)

,replication,n_candidates,n_SLO_ok,utility_range,allocation_range,eviction_min,eviction_max,mean_HP_queue_min,mean_HP_queue_max,surface_identified
0,R04,15,15,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False
1,R05,15,15,0.005779,0.005779,0.002290,0.003282,0.000000,0.002976,True
2,R06,15,15,0.010809,0.010809,0.001574,0.007769,0.000000,0.000000,True
3,R07,15,15,0.001124,0.001124,0.000308,0.004052,0.000000,0.000000,True
4,R08,15,15,0.008303,0.008303,0.004014,0.013690,0.178075,0.634127,True
5,R09,15,15,0.001785,0.001785,0.004249,0.021349,5.241270,7.154960,True


In [19]:
print("Configuração paralela:")
print(" MAX_WORKERS =", MAX_WORKERS)
print(" PARALLEL_INFO =", PARALLEL_INFO)

if (OUT/"checkpoint_runtime.csv").exists():
    cp_rt = pd.read_csv(
        OUT/"checkpoint_runtime.csv"
    )

    display(cp_rt)

    print(
        "Warm-up incremental medido nesta execução (h):",
        round(
            cp_rt["warmup_increment_s"].sum()/3600,
            2
        )
    )

if (OUT/"parallel_runtime_progress.csv").exists():
    pr = pd.read_csv(
        OUT/"parallel_runtime_progress.csv"
    )

    display(pr.tail())

    if len(pr):
        last = pr.iloc[-1]

        print(
            "Wall time dos branches nesta sessão (h):",
            round(
                last["wall_elapsed_s"]/3600,
                2
            )
        )

        print(
            "Tempo médio por branch único (min):",
            round(
                last["mean_successful_branch_s"]/60,
                2
            )
            if np.isfinite(
                last["mean_successful_branch_s"]
            )
            else np.nan
        )

if RESULT_CSV.exists():
    rr = pd.read_csv(RESULT_CSV)

    if "execution_mode" in rr.columns:
        print("\\nModos de execução:")
        display(
            rr["execution_mode"]
            .value_counts(dropna=False)
            .to_frame("n")
        )

Configuração paralela:
 MAX_WORKERS = 6
 PARALLEL_INFO = {'logical_cpus': 20, 'physical_cpus': 12, 'available_ram_gb': 49.53, 'cpu_limit': 11, 'ram_limit': 16}


,replication,checkpoint_time,warmup_increment_s,checkpoint_MB
0,R04,8467200,1582.415912,25.588008
1,R05,9676800,261.441782,25.794885
2,R06,10886400,125.894005,26.030251
3,R07,12096000,123.506052,26.884257
4,R08,13305600,115.014891,27.167892
5,R09,14515200,263.558009,27.431915


Warm-up incremental medido nesta execução (h): 0.69


,completed_unique_classes,total_unique_classes,wall_elapsed_s,mean_successful_branch_s,approx_eta_s,workers
85,86,90,6354.392521,436.162598,290.775065,6
86,87,90,6414.744546,434.111138,217.055569,6
87,88,90,6459.141513,432.311177,144.103726,6
88,89,90,6464.746106,430.391078,71.731846,6
89,90,90,6583.923962,428.794336,0.000000,6


Wall time dos branches nesta sessão (h): 1.83
Tempo médio por branch único (min): 7.15
\nModos de execução:


,n
execution_mode,
simulated,90


In [20]:
zip_out=ROOT/"outputs_fame_gpu_04a_v3_parallel.zip"

with zipfile.ZipFile(
    zip_out,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(
                p,
                arcname=f"outputs_fame_gpu_04a_v3_parallel/{p.name}"
            )

print("ZIP:",zip_out.resolve())
print("Envie este ZIP antes da transferência congelada para E.")

ZIP: C:\Users\tiago\Dropbox\Artigos\FAME\FAME\Artigo 2 - AOR\FAME_GPU\outputs_fame_gpu_04a_v3_parallel.zip
Envie este ZIP antes da transferência congelada para E.
